# Motion tracking

[Data](https://www.kaggle.com/datasets/smeschke/pedestrian-dataset?resource=download) - drone video of a roundabout


In [2]:
import cv2 as cv
import numpy as np

In [3]:
# Video file names
file1 = 'data/crosswalk.avi'
file2 = 'data/fourway.avi'
file3 = 'data/night.avi'

### Sparse optical flow

- Visualize trajectories of moving objects.
- Optional task: Identify each object and count them.
- Use following functions: `cv::goodFeaturesToTrack`, `cv::calcOpticalFlowPyrLK`


In [19]:
# Load the video
cap = cv.VideoCapture(file1)

scale = 0.7

# Define the parameters for goodFeaturesToTrack function
feature_params = {
    'maxCorners': 100,
    'qualityLevel': 0.3,
    'minDistance': 7,
    'blockSize': 7,
}

# Define the parameters for calcOpticalFlowPyrLK function
lk_params = {
    'winSize': (15, 15),
    'maxLevel': 2,
    'criteria': (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03),
}

# Take the first frame of the video
_, old_frame = cap.read()
old_gray = cv.cvtColor(old_frame, cv.COLOR_BGR2GRAY)

# Get the initial feature points to track
p0 = cv.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

# Create a mask image for drawing purposes
mask = np.zeros_like(old_frame)

while True:
    # Capture a frame from the video
    ret, new_frame = cap.read()

    if not ret:
        break

    # Convert the frame to grayscale
    new_gray = cv.cvtColor(new_frame, cv.COLOR_BGR2GRAY)

    # Calculate optical flow using calcOpticalFlowPyrLK function
    p1, st, err = cv.calcOpticalFlowPyrLK(old_gray, new_gray, p0, None, **lk_params)

    # Select good points
    good_new = p1[st==1]
    good_old = p0[st==1]

    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        mask = cv.line(mask, (a, b), (c, d), (0, 255, 0), 2)
        new_frame = cv.circle(new_frame, (a, b), 5, (0, 255, 0), -1)

    img = cv.add(new_frame, mask)

    # Display the resulting rescaled image
    img = cv.resize(img, (0, 0), fx=scale, fy=scale)
    cv.imshow('Sparse Optical Flow', img)

    # Wait for a key press
    if cv.waitKey(25) & 0xFF == ord('q'):
        break

    # Update the previous frame and points
    old_gray = new_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)

# Release the video capture and close all windows
cap.release()
cv.destroyAllWindows()


### Dense optical flow

- Identify moving objects in video and draw green rectangle around them.
- Use downsampled video for this task if necessary for easier processing.
- Use following functions: `cv::calcOpticalFlowFarneback`
